# Notebook 07 — Statistical Time-Series Models

This notebook establishes classical statistical forecasting baselines for the S&P 500.

**Input**
`data/raw/sp500_1950_present.csv`

**Models**
- Naive / persistence baseline
- Historical-mean baseline
- Drift baseline
- AutoRegressive (AR) model
- ARIMA
- Exponential Smoothing / Holt trend model
- Optional GARCH volatility model when the `arch` package is available

**Evaluation**
- Chronological train/test split
- RMSE
- MAE
- MAPE where meaningful
- Directional accuracy
- Residual diagnostics

This notebook is a classical statistical baseline. It does not perform random train/test splitting and does not use future observations to construct predictors.


## 1. Imports

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
)

from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.graphics.tsaplots import plot_acf

warnings.filterwarnings("ignore")

print("Imports loaded successfully.")


## 2. Configuration and Paths

In [ ]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "data").exists():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/mnt/data/quant-trading-research"),
    ]
    for candidate in candidates:
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            PROJECT_ROOT = candidate
            break

MASTER_PATH = PROJECT_ROOT / "data" / "raw" / "sp500_1950_present.csv"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"
TABLE_DIR = PROJECT_ROOT / "reports" / "tables"
REPORT_DIR = PROJECT_ROOT / "reports" / "generated"

for path in [INTERIM_DIR, FIGURE_DIR, TABLE_DIR, REPORT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

EXPECTED_COLUMNS = [
    "Date", "Open", "High", "Low", "Close", "Adj.Close", "Volume"
]

TEST_FRACTION = 0.20

print(f"Master dataset: {MASTER_PATH}")


## 3. Load and Validate the Master Dataset

In [ ]:
if not MASTER_PATH.exists():
    raise FileNotFoundError(
        f"Master dataset not found: {MASTER_PATH}. "
        "Run the earlier notebooks first."
    )

df = pd.read_csv(MASTER_PATH, low_memory=False)

if list(df.columns) != EXPECTED_COLUMNS:
    raise ValueError(
        f"Unexpected schema. Expected {EXPECTED_COLUMNS}; "
        f"received {list(df.columns)}"
    )

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

for column in EXPECTED_COLUMNS[1:]:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df = df.sort_values("Date").reset_index(drop=True)

if df["Date"].isna().any():
    raise ValueError("Invalid dates detected.")

if df["Date"].duplicated().any():
    raise ValueError("Duplicate dates detected.")

if df[["Open", "High", "Low", "Close", "Adj.Close", "Volume"]].isna().any().any():
    raise ValueError("Missing OHLCV values detected.")

print(f"Rows: {len(df):,}")
print(f"Date range: {df['Date'].min().date()} → {df['Date'].max().date()}")


## 4. Construct the Modeling Series

The primary forecasting target is the **daily Close level**.

Daily log returns are also retained for volatility modeling.

The raw master dataset is never overwritten.


In [ ]:
series = (
    df[["Date", "Close"]]
    .set_index("Date")["Close"]
    .astype(float)
)

returns = np.log(series).diff().dropna()

print(f"Close observations: {len(series):,}")
print(f"Log-return observations: {len(returns):,}")
display(series.head())


## 5. Chronological Train/Test Split

The final 20% of observations is reserved as a chronological holdout.

No observations from the test period are used when fitting the forecasting models.


In [ ]:
split_idx = int(len(series) * (1 - TEST_FRACTION))

train = series.iloc[:split_idx].copy()
test = series.iloc[split_idx:].copy()

print(f"Train observations: {len(train):,}")
print(f"Test observations: {len(test):,}")
print(f"Train end: {train.index.max().date()}")
print(f"Test start: {test.index.min().date()}")
print(f"Test end: {test.index.max().date()}")


## 6. Holdout Visualization

In [ ]:
fig = plt.figure(figsize=(14, 6))
plt.plot(train.index, train, label="Train")
plt.plot(test.index, test, label="Test")
plt.axvline(test.index[0], linestyle="--", label="Test start")
plt.title("Chronological Statistical-Model Train/Test Split")
plt.xlabel("Date")
plt.ylabel("S&P 500 Close")
plt.legend()
plt.grid(True, alpha=0.25)
plt.tight_layout()

path = FIGURE_DIR / "sp500_statistical_model_train_test_split.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved: {path}")


## 7. Evaluation Functions

In [ ]:
def evaluate_forecast(actual, predicted, model_name):
    actual = pd.Series(actual).astype(float)
    predicted = pd.Series(predicted, index=actual.index).astype(float)

    errors = actual - predicted

    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mae = mean_absolute_error(actual, predicted)

    nonzero_actual = actual.replace(0, np.nan)
    mape = (
        (errors.abs() / nonzero_actual.abs())
        .dropna()
        .mean() * 100
    )

    if len(actual) > 1:
        actual_direction = np.sign(actual.diff())
        predicted_direction = np.sign(predicted.diff())

        direction_mask = (
            actual_direction.notna()
            & predicted_direction.notna()
            & (actual_direction != 0)
        )

        directional_accuracy = (
            (actual_direction[direction_mask] ==
             predicted_direction[direction_mask])
            .mean()
        )
    else:
        directional_accuracy = np.nan

    return {
        "model": model_name,
        "RMSE": float(rmse),
        "MAE": float(mae),
        "MAPE_percent": float(mape),
        "Directional_Accuracy": float(directional_accuracy),
    }


## 8. Naive Persistence Baseline

The next close is predicted as the most recently observed close.

This is the essential baseline for a financial price-level forecasting problem.


In [ ]:
naive_pred = pd.Series(
    train.iloc[-1],
    index=test.index,
    name="prediction"
)

naive_metrics = evaluate_forecast(
    test,
    naive_pred,
    "Naive Persistence"
)

print(naive_metrics)


## 9. Historical-Mean Baseline

In [ ]:
mean_pred = pd.Series(
    train.mean(),
    index=test.index,
    name="prediction"
)

mean_metrics = evaluate_forecast(
    test,
    mean_pred,
    "Historical Mean"
)

print(mean_metrics)


## 10. Drift Baseline

The drift model extrapolates the average historical change from the first to the latest training observation.


In [ ]:
n_train = len(train)

average_drift = (
    (train.iloc[-1] - train.iloc[0])
    / max(n_train - 1, 1)
)

drift_steps = np.arange(1, len(test) + 1)

drift_pred = pd.Series(
    train.iloc[-1] + average_drift * drift_steps,
    index=test.index,
    name="prediction"
)

drift_metrics = evaluate_forecast(
    test,
    drift_pred,
    "Drift"
)

print(drift_metrics)


## 11. AutoRegressive Model

An AutoReg model uses lagged historical close levels.

A moderate lag length is used as a classical baseline. The model is fitted only on the training set.


In [ ]:
ar_lags = 20

ar_model = AutoReg(
    train,
    lags=ar_lags,
    trend="ct",
    old_names=False,
).fit()

ar_pred = ar_model.predict(
    start=len(train),
    end=len(train) + len(test) - 1,
    dynamic=False,
)

ar_pred.index = test.index

ar_metrics = evaluate_forecast(
    test,
    ar_pred,
    f"AutoReg({ar_lags})"
)

print(ar_metrics)


## 12. ARIMA Model

ARIMA is fitted to the close-price level series.

The selected order is intentionally kept modest for a reproducible classical baseline.


In [ ]:
arima_order = (1, 1, 1)

arima_model = ARIMA(
    train,
    order=arima_order,
    enforce_stationarity=False,
    enforce_invertibility=False,
).fit()

arima_pred = arima_model.forecast(
    steps=len(test)
)

arima_pred.index = test.index

arima_metrics = evaluate_forecast(
    test,
    arima_pred,
    f"ARIMA{arima_order}"
)

print(arima_metrics)


## 13. Holt Exponential Smoothing

Holt's method models the level and trend of the price series.

It is another classical baseline against which more complex models should be compared.


In [ ]:
holt_model = ExponentialSmoothing(
    train,
    trend="add",
    damped_trend=True,
    initialization_method="estimated",
).fit(
    optimized=True
)

holt_pred = holt_model.forecast(
    len(test)
)

holt_pred.index = test.index

holt_metrics = evaluate_forecast(
    test,
    holt_pred,
    "Holt Damped Trend"
)

print(holt_metrics)


## 14. Collect Point-Forecast Results

In [ ]:
metrics = pd.DataFrame([
    naive_metrics,
    mean_metrics,
    drift_metrics,
    ar_metrics,
    arima_metrics,
    holt_metrics,
])

metrics = metrics.sort_values(
    "RMSE",
    ascending=True
).reset_index(drop=True)

display(metrics)

metrics.to_csv(
    TABLE_DIR / "sp500_statistical_model_comparison.csv",
    index=False
)


## 15. Forecast Comparison Plot

In [ ]:
fig = plt.figure(figsize=(15, 7))

plt.plot(
    test.index,
    test,
    label="Actual"
)

plt.plot(
    test.index,
    naive_pred,
    label="Naive"
)

plt.plot(
    test.index,
    ar_pred,
    label="AutoReg"
)

plt.plot(
    test.index,
    arima_pred,
    label="ARIMA"
)

plt.plot(
    test.index,
    holt_pred,
    label="Holt"
)

plt.title("Statistical Model Forecast Comparison")
plt.xlabel("Date")
plt.ylabel("S&P 500 Close")
plt.legend()
plt.grid(True, alpha=0.25)
plt.tight_layout()

path = FIGURE_DIR / "sp500_statistical_model_forecasts.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved: {path}")


## 16. Zoomed Forecast Comparison

The final section of the holdout is shown separately to make short-term forecast behavior easier to inspect.


In [ ]:
zoom_n = min(250, len(test))

fig = plt.figure(figsize=(15, 7))

plt.plot(
    test.index[-zoom_n:],
    test.iloc[-zoom_n:],
    label="Actual"
)

plt.plot(
    ar_pred.index[-zoom_n:],
    ar_pred.iloc[-zoom_n:],
    label="AutoReg"
)

plt.plot(
    arima_pred.index[-zoom_n:],
    arima_pred.iloc[-zoom_n:],
    label="ARIMA"
)

plt.plot(
    holt_pred.index[-zoom_n:],
    holt_pred.iloc[-zoom_n:],
    label="Holt"
)

plt.title("Statistical Model Forecasts — Holdout Tail")
plt.xlabel("Date")
plt.ylabel("S&P 500 Close")
plt.legend()
plt.grid(True, alpha=0.25)
plt.tight_layout()

path = FIGURE_DIR / "sp500_statistical_model_forecasts_zoomed.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved: {path}")


## 17. Forecast Residuals

Residuals are calculated as actual minus predicted.

The best model is selected by holdout RMSE only for diagnostic comparison; this does not make the holdout a tuning set.


In [ ]:
best_model_name = metrics.iloc[0]["model"]

prediction_map = {
    "Naive Persistence": naive_pred,
    "Historical Mean": mean_pred,
    "Drift": drift_pred,
    f"AutoReg({ar_lags})": ar_pred,
    f"ARIMA{arima_order}": arima_pred,
    "Holt Damped Trend": holt_pred,
}

best_pred = prediction_map[best_model_name]

residuals = (
    test - best_pred
).rename("residual")

print(f"Best holdout RMSE model: {best_model_name}")
display(residuals.describe())


## 18. Residual Distribution

In [ ]:
fig = plt.figure(figsize=(12, 6))
plt.hist(residuals.dropna(), bins=100)
plt.title(f"Residual Distribution — {best_model_name}")
plt.xlabel("Residual")
plt.ylabel("Frequency")
plt.grid(True, alpha=0.25)
plt.tight_layout()

path = FIGURE_DIR / "sp500_statistical_model_residual_distribution.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved: {path}")


## 19. Residual Autocorrelation

In [ ]:
fig = plt.figure(figsize=(12, 6))
plot_acf(
    residuals.dropna(),
    lags=40,
    ax=plt.gca()
)
plt.title(f"Residual Autocorrelation — {best_model_name}")
plt.tight_layout()

path = FIGURE_DIR / "sp500_statistical_model_residual_acf.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved: {path}")


## 20. Ljung–Box Residual Diagnostic

In [ ]:
lb = acorr_ljungbox(
    residuals.dropna(),
    lags=[10, 20, 40],
    return_df=True,
)

display(lb)

lb.to_csv(
    TABLE_DIR / "sp500_statistical_model_ljung_box.csv"
)


## 21. Residual Volatility Diagnostics

In [ ]:
residual_volatility = pd.Series(
    residuals,
    index=residuals.index,
    name="residual"
)

residual_rolling_std = residual_volatility.rolling(21).std()

fig = plt.figure(figsize=(14, 6))
plt.plot(
    residual_rolling_std.index,
    residual_rolling_std
)
plt.title(f"21-Day Rolling Residual Volatility — {best_model_name}")
plt.xlabel("Date")
plt.ylabel("Residual Standard Deviation")
plt.grid(True, alpha=0.25)
plt.tight_layout()

path = FIGURE_DIR / "sp500_statistical_model_residual_volatility.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved: {path}")


## 22. Optional GARCH Volatility Model

GARCH is applied to daily log returns rather than the price level.

The package `arch` is optional because it is not part of the minimum classical-model stack.

If unavailable, the notebook records the dependency status and continues.


In [ ]:
garch_available = False
garch_result = None
garch_forecast = None

try:
    from arch import arch_model
    garch_available = True
    print("arch package available.")
except ImportError:
    print(
        "arch package is not installed. "
        "GARCH analysis will be skipped."
    )


## 23. Fit GARCH(1,1) if Available

Returns are multiplied by 100 for numerical scaling.


In [ ]:
if garch_available:
    train_returns = np.log(train).diff().dropna() * 100

    garch = arch_model(
        train_returns,
        mean="Constant",
        vol="GARCH",
        p=1,
        q=1,
        dist="normal",
        rescale=False,
    )

    garch_result = garch.fit(
        disp="off"
    )

    print(garch_result.summary())

    garch_forecast = garch_result.forecast(
        horizon=len(test),
        reindex=False,
    )

    variance_forecast = (
        np.asarray(
            garch_forecast.variance.iloc[-1]
        )
        / 100**2
    )

    garch_volatility = pd.Series(
        np.sqrt(variance_forecast) * np.sqrt(252),
        index=test.index,
        name="annualized_forecast_volatility",
    )

    display(garch_volatility.head())

    garch_volatility.to_csv(
        TABLE_DIR / "sp500_garch_forecast_volatility.csv"
    )
else:
    print("GARCH skipped.")


## 24. GARCH Forecast Visualization

In [ ]:
if garch_available:
    fig = plt.figure(figsize=(14, 6))
    plt.plot(
        garch_volatility.index,
        garch_volatility
    )
    plt.title("GARCH(1,1) Forecast Annualized Volatility")
    plt.xlabel("Date")
    plt.ylabel("Annualized Volatility")
    plt.grid(True, alpha=0.25)
    plt.tight_layout()

    path = FIGURE_DIR / "sp500_garch_forecast_volatility.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()

    print(f"Saved: {path}")
else:
    print("No GARCH figure generated because arch is unavailable.")


## 25. Statistical Model Ranking

In [ ]:
metrics_ranked = metrics.copy()

metrics_ranked["RMSE_rank"] = metrics_ranked["RMSE"].rank(
    method="min"
)

metrics_ranked["MAE_rank"] = metrics_ranked["MAE"].rank(
    method="min"
)

metrics_ranked["Directional_rank"] = (
    metrics_ranked["Directional_Accuracy"]
    .rank(
        method="min",
        ascending=False
    )
)

display(metrics_ranked)

metrics_ranked.to_csv(
    TABLE_DIR / "sp500_statistical_model_ranking.csv",
    index=False
)


## 26. Statistical Model Report

In [ ]:
statistical_report = {
    "dataset": {
        "rows": int(len(df)),
        "start": df["Date"].min().strftime("%Y-%m-%d"),
        "end": df["Date"].max().strftime("%Y-%m-%d"),
    },
    "split": {
        "train_rows": int(len(train)),
        "test_rows": int(len(test)),
        "train_end": train.index.max().strftime("%Y-%m-%d"),
        "test_start": test.index.min().strftime("%Y-%m-%d"),
        "test_end": test.index.max().strftime("%Y-%m-%d"),
    },
    "models": metrics.to_dict(orient="records"),
    "best_holdout_model_by_rmse": best_model_name,
    "residual_diagnostics": lb.reset_index().to_dict(
        orient="records"
    ),
    "garch_available": garch_available,
    "methodological_note": (
        "The holdout is chronological. No random shuffling is used. "
        "The reported model comparison is a classical statistical baseline "
        "and is not a substitute for walk-forward validation."
    ),
}

report_path = REPORT_DIR / "sp500_statistical_models_report.json"

report_path.write_text(
    json.dumps(statistical_report, indent=2, default=str),
    encoding="utf-8"
)

print(json.dumps(statistical_report, indent=2, default=str))
print(f"\nSaved: {report_path}")


## 27. Save Statistical Research Dataset

In [ ]:
statistical_output = INTERIM_DIR / "sp500_statistical_model_inputs.parquet"

model_input = df[
    [
        "Date",
        "Open",
        "High",
        "Low",
        "Close",
        "Adj.Close",
        "Volume",
    ]
].copy()

model_input["return_1d"] = model_input["Close"].pct_change()
model_input["log_return_1d"] = np.log(model_input["Close"]).diff()

model_input.to_parquet(
    statistical_output,
    index=False
)

print(f"Saved: {statistical_output}")


## 28. Final Master Dataset Integrity Check

The statistical-model notebook must not alter the raw acquisition CSV.


In [ ]:
master_check = pd.read_csv(
    MASTER_PATH,
    low_memory=False
)

assert list(master_check.columns) == EXPECTED_COLUMNS
assert len(master_check) == len(df)

master_dates = pd.to_datetime(
    master_check["Date"],
    errors="coerce"
)

assert master_dates.notna().all()
assert master_dates.is_unique
assert master_dates.is_monotonic_increasing

print("Raw master dataset integrity after statistical modeling: PASS")
print(f"Master rows: {len(master_check):,}")


# Notebook 07 Complete

Notebook 07 establishes the classical statistical baseline.

### Models
- Naive persistence
- Historical mean
- Drift
- AutoRegressive model
- ARIMA
- Holt damped trend
- Optional GARCH(1,1) volatility model

### Evaluation
- Chronological holdout
- RMSE
- MAE
- MAPE
- Directional accuracy
- Residual distribution
- Residual autocorrelation
- Ljung–Box diagnostics
- Residual volatility

### Important methodological boundary

This is a baseline modeling notebook. It does not claim that the best holdout model is a deployable trading strategy. Later notebooks should use leakage-controlled walk-forward validation and proper trading-cost-aware backtesting.

**Next notebook:** Notebook 08 — Machine Learning Models.

Run Notebook 07 from top to bottom and verify the outputs before proceeding.
